In [1]:
#DECODELABS INTERNSHIP - DATA SCIENCE PROJECT 2 - Amashi Fernando
#Supervised Learning (Fraud Detection Pipeline)
#Dataset: Credit cards

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve,
    precision_recall_curve, ConfusionMatrixDisplay
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [4]:
# Config

RANDOM_STATE = 42
TEST_SIZE = 0.20

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [5]:
# Load Data

df = pd.read_csv("Dataset-creditcard.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values per column:\n{df.isnull().sum().sum()} total missing values")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Drop exact duplicate rows if any
df = df.drop_duplicates()
print(f"Shape after dropping duplicates: {df.shape}")

Shape: (284807, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Missing values per column:
0 total missing values

Duplicate rows: 1081
Shape after dropping duplicates: (283726, 31)


In [6]:
# Exploratary Data Analysis (EDA)

class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100
print(f"\nClass distribution:\n{class_counts}")
print(f"\nClass percentage:\n{class_pct.round(4)}")
print(f"\nFraud rate: {class_pct[1]:.4f}%  |  Legitimate rate: {class_pct[0]:.4f}%")

# --- Plot 1: Class imbalance bar chart ---
plt.figure()
ax = sns.countplot(x="Class", data=df, palette=["#2E86AB", "#E94F37"])
plt.title("Class Distribution: Legitimate (0) vs Fraudulent (1)")
plt.xlabel("Class")
plt.ylabel("Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig("01_class_distribution.png", dpi=120)
plt.close()
print("Saved: 01_class_distribution.png")

# --- Plot 2: Transaction Amount distribution by class ---
plt.figure()
sns.histplot(data=df, x="Amount", hue="Class", bins=50, log_scale=(False, True),
             palette=["#2E86AB", "#E94F37"], element="step")
plt.title("Transaction Amount Distribution by Class")
plt.xlabel("Amount")
plt.ylabel("Count (log scale)")
plt.xlim(0, 2500)
plt.tight_layout()
plt.savefig("02_amount_distribution.png", dpi=120)
plt.close()
print("Saved: 02_amount_distribution.png")

# --- Plot 3: Correlation heatmap (PCA components + Amount/Time vs Class) ---
plt.figure(figsize=(14, 10))
corr = df.corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, linewidths=0.3)
plt.title("Correlation Heatmap (All Features)")
plt.tight_layout()
plt.savefig("03_correlation_heatmap.png", dpi=120)
plt.close()
print("Saved: 03_correlation_heatmap.png")

# --- Plot 4: Time distribution by class ---
plt.figure()
sns.histplot(data=df, x="Time", hue="Class", bins=50,
             palette=["#2E86AB", "#E94F37"], element="step", common_norm=False, stat="density")
plt.title("Transaction Time Distribution by Class (Normalized)")
plt.xlabel("Time (seconds since first transaction)")
plt.tight_layout()
plt.savefig("04_time_distribution.png", dpi=120)
plt.close()
print("Saved: 04_time_distribution.png")


Class distribution:
Class
0    283253
1       473
Name: count, dtype: int64

Class percentage:
Class
0    99.8333
1     0.1667
Name: proportion, dtype: float64

Fraud rate: 0.1667%  |  Legitimate rate: 99.8333%


/tmp/ipykernel_1641/2349404479.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(x="Class", data=df, palette=["#2E86AB", "#E94F37"])


Saved: 01_class_distribution.png
Saved: 02_amount_distribution.png
Saved: 03_correlation_heatmap.png
Saved: 04_time_distribution.png


In [7]:
# TRAIN / TEST SPLIT  (Before any scaling)

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train shape: {X_train.shape} | Fraud count: {y_train.sum()} "
      f"({y_train.mean()*100:.4f}%)")
print(f"Test  shape: {X_test.shape} | Fraud count: {y_test.sum()} "
      f"({y_test.mean()*100:.4f}%)")
print("\n--> Test set is UNTOUCHED by scaling/SMOTE and reflects real-world imbalance.")

Train shape: (226980, 30) | Fraud count: 378 (0.1665%)
Test  shape: (56746, 30) | Fraud count: 95 (0.1674%)

--> Test set is UNTOUCHED by scaling/SMOTE and reflects real-world imbalance.


In [8]:
# PIPELINE 1: LOGISTIC REGRESSION

lr_pipeline = ImbPipeline(steps=[
    ("scaler", StandardScaler()),                      # LR is scale-sensitive
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

lr_param_grid = {
    "smote__k_neighbors": [3, 5],
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
    "classifier__penalty": ["l2"],
    "classifier__solver": ["lbfgs"]
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring="roc_auc",      # optimize for ROC-AUC, never accuracy
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1
)

print("Fitting GridSearchCV for Logistic Regression... (SMOTE applied only on training folds)")
lr_grid.fit(X_train, y_train)

print(f"\nBest LR params: {lr_grid.best_params_}")
print(f"Best LR CV ROC-AUC: {lr_grid.best_score_:.4f}")

best_lr = lr_grid.best_estimator_

Fitting GridSearchCV for Logistic Regression... (SMOTE applied only on training folds)
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best LR params: {'classifier__C': 0.01, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs', 'smote__k_neighbors': 3}
Best LR CV ROC-AUC: 0.9816


In [12]:
# PIPELINE 2: RANDOM FOREST  (no scaler needed, no GridSearchCV)

rf_pipeline = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

print("Fitting Random Forest pipeline (single fit)")
rf_pipeline.fit(X_train, y_train)

best_rf = rf_pipeline

Fitting Random Forest pipeline (single fit)


In [15]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

def evaluate_model(name, model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted pipeline on the test set with strict fraud-relevant metrics."""
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n--- {name} ---")
    print(f"Precision : {precision:.4f}  (When we flag fraud, how often are we right?)")
    print(f"Recall    : {recall:.4f}  (Of all actual fraud, how much did we catch?)")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'])}")

    return {
        "name": name, "precision": precision, "recall": recall,
        "f1": f1, "roc_auc": roc_auc, "y_proba": y_proba,
        "y_pred": y_pred, "confusion_matrix": cm
    }

In [17]:
print("Fitting Random Forest pipeline (single fit)")
rf_pipeline.fit(X_train, y_train)

best_rf = rf_pipeline

# Evaluate immediately
results_rf = evaluate_model("Random Forest (single fit)", best_rf, X_test, y_test)

Fitting Random Forest pipeline (single fit)

--- Random Forest (single fit) ---
Precision : 0.6016  (When we flag fraud, how often are we right?)
Recall    : 0.8105  (Of all actual fraud, how much did we catch?)
F1-Score  : 0.6906
ROC-AUC   : 0.9729
Confusion Matrix:
[[56600    51]
 [   18    77]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56651
       Fraud       0.60      0.81      0.69        95

    accuracy                           1.00     56746
   macro avg       0.80      0.90      0.84     56746
weighted avg       1.00      1.00      1.00     56746



In [18]:
# Final Evaluation on untouched test set

def evaluate_model(name, model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted pipeline on the test set with strict fraud-relevant metrics."""
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n--- {name} ---")
    print(f"Precision : {precision:.4f}  (When we flag fraud, how often are we right?)")
    print(f"Recall    : {recall:.4f}  (Of all actual fraud, how much did we catch?)")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud'])}")

    return {
        "name": name, "precision": precision, "recall": recall,
        "f1": f1, "roc_auc": roc_auc, "y_proba": y_proba,
        "y_pred": y_pred, "confusion_matrix": cm
    }

results_lr = evaluate_model("Logistic Regression (tuned)", best_lr, X_test, y_test)
results_rf = evaluate_model("Random Forest (single fit)", best_rf, X_test, y_test)


--- Logistic Regression (tuned) ---
Precision : 0.0532  (When we flag fraud, how often are we right?)
Recall    : 0.8737  (Of all actual fraud, how much did we catch?)
F1-Score  : 0.1002
ROC-AUC   : 0.9634
Confusion Matrix:
[[55173  1478]
 [   12    83]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      0.97      0.99     56651
       Fraud       0.05      0.87      0.10        95

    accuracy                           0.97     56746
   macro avg       0.53      0.92      0.54     56746
weighted avg       1.00      0.97      0.99     56746


--- Random Forest (single fit) ---
Precision : 0.6016  (When we flag fraud, how often are we right?)
Recall    : 0.8105  (Of all actual fraud, how much did we catch?)
F1-Score  : 0.6906
ROC-AUC   : 0.9729
Confusion Matrix:
[[56600    51]
 [   18    77]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     56651
     

In [19]:
# Visualize Results : Confusion matrices, ROC, Precision-Recall curves

# --- Confusion matrices ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, res in zip(axes, [results_lr, results_rf]):
    disp = ConfusionMatrixDisplay(confusion_matrix=res["confusion_matrix"],
                                   display_labels=["Legitimate", "Fraud"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(res["name"])
plt.tight_layout()
plt.savefig("05_confusion_matrices.png", dpi=120)
plt.close()
print("Saved: 05_confusion_matrices.png")

# --- ROC curves ---
plt.figure()
for res, color in zip([results_lr, results_rf], ["#2E86AB", "#E94F37"]):
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    plt.plot(fpr, tpr, label=f"{res['name']} (AUC = {res['roc_auc']:.4f})", color=color)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("06_roc_curves.png", dpi=120)
plt.close()
print("Saved: 06_roc_curves.png")

# --- Precision-Recall curves ---
plt.figure()
for res, color in zip([results_lr, results_rf], ["#2E86AB", "#E94F37"]):
    prec, rec, _ = precision_recall_curve(y_test, res["y_proba"])
    plt.plot(rec, prec, label=res["name"], color=color)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")
plt.legend(loc="lower left")
plt.tight_layout()
plt.savefig("07_precision_recall_curves.png", dpi=120)
plt.close()
print("Saved: 07_precision_recall_curves.png")

# --- Random Forest feature importance ---
rf_classifier = best_rf.named_steps["classifier"]
importances = pd.Series(rf_classifier.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=False).head(15)

plt.figure()
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Top 15 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("08_feature_importance.png", dpi=120)
plt.close()
print("Saved: 08_feature_importance.png")

Saved: 05_confusion_matrices.png
Saved: 06_roc_curves.png
Saved: 07_precision_recall_curves.png


/tmp/ipykernel_1641/3418334832.py:50: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importances.values, y=importances.index, palette="viridis")


Saved: 08_feature_importance.png


In [20]:
# Final Model Comparison Summary

summary_df = pd.DataFrame([
    {"Model": results_lr["name"], "Precision": results_lr["precision"],
     "Recall": results_lr["recall"], "F1-Score": results_lr["f1"],
     "ROC-AUC": results_lr["roc_auc"]},
    {"Model": results_rf["name"], "Precision": results_rf["precision"],
     "Recall": results_rf["recall"], "F1-Score": results_rf["f1"],
     "ROC-AUC": results_rf["roc_auc"]},
]).round(4)

print(f"\n{summary_df.to_string(index=False)}")
summary_df.to_csv("model_comparison_summary.csv", index=False)
print("\nSaved: model_comparison_summary.csv")



                      Model  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression (tuned)     0.0532  0.8737    0.1002   0.9634
 Random Forest (single fit)     0.6016  0.8105    0.6906   0.9729

Saved: model_comparison_summary.csv


In [22]:
# Threshold Sensitivity Analysis — Logistic Regression
# At the default threshold (0.5), LR has very low precision (many false alarms).
# Raising the threshold makes LR more conservative

for t in [0.3, 0.5, 0.7, 0.9]:
    print(f"\n{'='*50}")
    print(f"THRESHOLD = {t}")
    print(f"{'='*50}")
    _ = evaluate_model(f"Logistic Regression (threshold={t})", best_lr, X_test, y_test, threshold=t)


THRESHOLD = 0.3

--- Logistic Regression (threshold=0.3) ---
Precision : 0.0266  (When we flag fraud, how often are we right?)
Recall    : 0.8842  (Of all actual fraud, how much did we catch?)
F1-Score  : 0.0516
ROC-AUC   : 0.9634
Confusion Matrix:
[[53576  3075]
 [   11    84]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      0.95      0.97     56651
       Fraud       0.03      0.88      0.05        95

    accuracy                           0.95     56746
   macro avg       0.51      0.91      0.51     56746
weighted avg       1.00      0.95      0.97     56746


THRESHOLD = 0.5

--- Logistic Regression (threshold=0.5) ---
Precision : 0.0532  (When we flag fraud, how often are we right?)
Recall    : 0.8737  (Of all actual fraud, how much did we catch?)
F1-Score  : 0.1002
ROC-AUC   : 0.9634
Confusion Matrix:
[[55173  1478]
 [   12    83]]

Classification Report:
              precision    recall  f1-score   support

  Legiti

* In this project, I built a fraud detection pipeline on the Kaggle credit card dataset, which has 284,807 transactions but only 0.17% labeled as fraud.
* Because of this extreme imbalance, I avoided using accuracy as a metric and instead evaluated my models using Precision, Recall, F1-score, and ROC-AUC.
* I used SMOTE to balance the training data by generating synthetic fraud examples, making sure to apply it only after the train-test split and only inside an imblearn.Pipeline to avoid data leakage.
* I trained two models: a Logistic Regression model (tuned using GridSearchCV with feature scaling) and a Random Forest model (trained with a single sensible configuration instead of a full grid search, since tuning it exhaustively was too slow on this dataset size).
* Comparing both models on the untouched test set let me see the tradeoffs between a simpler tuned model and a more complex ensemble model.
* Through this project, I learned how to handle imbalanced data correctly, how to prevent leakage during preprocessing, and how to choose evaluation metrics that actually matter for a fraud detection problem.
* A threshold sensitivity analysis on Logistic Regression showed that raising the decision threshold from 0.5 to 0.9 improved precision more than threefold (0.053 → 0.200) while recall dropped only slightly (0.874 → 0.842), more than tripling the F1-score (0.100 → 0.323).
* This confirmed that ROC-AUC alone doesn't reveal the best operating point for a model — F1-score at different thresholds does.
* Even after this tuning, Random Forest's single-fit F1-score (0.691) remained substantially higher than Logistic Regression's best-threshold F1-score (0.323), reinforcing Random Forest as the stronger model for this task, though threshold tuning would be a necessary step before deploying Logistic Regression in practice

#### Amashi Fernando
Data Science Intern

DecodeLabs Internship (June 10 - July 10 2026)
